# Practical 6: Dataset Balancing & Stratified Partitioning for Machine Learning
**Course:** Python for Data Science (`BE05000231_PDS`)  
**Semester:** 5th Semester B.E. / B.Tech  
**Topic:** Mitigation of Extreme Class Imbalance, Stratified Train/Test Partitioning, and k-NN SMOTE Manifold Interpolation  
**Pipeline Input:** Practical 5 Feature-Engineered Dataset (`featured_logs.parquet` / `.csv` — 300,010 records)  
**External Benchmark Input:** Elastic Production Telemetry (`elastic_featured_logs.csv` — 10,000 records)  

---

## 1. AIM & OBJECTIVES
### Aim
To handle severe class imbalance across multi-class cyber telemetry by systematically evaluating Random Undersampling (RUS), Random Oversampling (ROS), and SMOTE strictly on an isolated training split, while preserving 100% test partition purity for authentic real-world model evaluation.

### Objectives
1. **Pipeline Continuity:** Ingest the 30-feature matrix with anomaly scores generated in Practical 5 (`featured_logs.parquet`).
2. **Zero-Leakage Split Protocol:** Execute a stratified 80% / 20% train-test split BEFORE applying any resampling, locking the 60,002 test records completely untouched.
3. **Methodology Comparison:** Compare Random Undersampling (RUS), Random Oversampling (ROS), and SMOTE across runtime, distribution entropy, and variance preservation.
4. **Equalized SMOTE Synthesis:** Synthesize mathematically novel feature vectors along k-NN manifolds ($k=5$), establishing a perfectly balanced 165,000-record training dataset (15,000 records per class across all 11 classes).
5. **High-Performance Serialization:** Export balanced training matrices and locked test partitions to CSV and Snappy-compressed Apache Parquet.
6. **Diagnostic Visualizations:** Plot original imbalance distributions, train/test alignment, technique comparisons, and 2D PCA projections of SMOTE manifolds.

---
## 2. THEORETICAL FOUNDATION: THE CLASS IMBALANCE PARADOX

### 2.1 The Accuracy Fallacy in Cybersecurity
In raw cyber telemetry, legitimate benign traffic dominates ($68.7\%$ to $99.9\%$), while specific attack vectors (`command_injection`, `sqli`) represent tiny minority fractions. If an algorithm is trained on raw imbalanced distributions:
- A trivial dummy classifier predicting "Benign" on every request achieves **$68.7\%$ raw accuracy**.
- However, its **Recall on cyberattacks is $0.0\%$**, allowing intrusions to pass completely undetected.

### 2.2 The Golden Rule of Resampling: Pre-Split Isolation
A catastrophic flaw in naive machine learning workflows is applying oversampling/SMOTE **before** splitting data:
$$\text{Incorrect: } \text{Raw Data} \xrightarrow{\text{SMOTE}} \text{Resampled Data} \xrightarrow{\text{Split}} \text{Train} + \text{Test} \quad (\text{FATAL LEAKAGE!})$$
When done this way, synthetic samples derived from test-set neighbors leak into the training fold, yielding artificially inflated metrics that fail completely in production.

$$\text{Enterprise Standard: } \text{Raw Data} \xrightarrow{\text{Stratified Split}} \text{Train (80\%)} \xrightarrow{\text{SMOTE}} \text{Balanced Train} \quad \& \quad \text{Test (20\% Locked)}$$

### 2.3 SMOTE Mathematics: Manifold Interpolation
Instead of duplicating rows (Random Oversampling), SMOTE finds the $k$-nearest neighbors of each minority class instance $x_i$ and generates a synthetic instance $x_{\text{new}}$ along the connecting line segment:

$$x_{\text{new}} = x_i + \lambda \cdot (x_{zi} - x_i), \quad \lambda \sim U(0, 1)$$

Where $x_{zi}$ is a randomly chosen neighbor from the $k$-nearest neighbors ($k=5$) of $x_i$ in Euclidean feature space.

---
## 3. SETUP & ENVIRONMENT
We import essential data science libraries, scikit-learn, and imbalanced-learn.

In [ ]:
import sys
import os
import time
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import RandomOverSampler, SMOTE

# Directory resolution
PRACTICAL_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
WORKSPACE_DIR = PRACTICAL_DIR.parent

sys.path.insert(0, str(WORKSPACE_DIR))
from src.features.feature_definitions import CANONICAL_FEATURE_NAMES

P5_PROCESSED = WORKSPACE_DIR / 'Practical 5' / 'data' / 'processed'
PROCESSED_DIR = PRACTICAL_DIR / 'data' / 'processed'
PLOTS_DIR = PRACTICAL_DIR / 'outputs' / 'plots'
REPORTS_DIR = PRACTICAL_DIR / 'outputs' / 'reports'

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

FEATURE_COLUMNS = CANONICAL_FEATURE_NAMES + ['anomaly_score']
print(f"Target Feature Vector: {len(FEATURE_COLUMNS)} dimensions")

---
## 4. INGESTION & ZERO-LEAKAGE STRATIFIED 80/20 PARTITIONING
We ingest the 300,010 records from Practical 5 and immediately partition them into 80% train and 20% test before touching any resampling algorithm.

In [ ]:
pq_path = P5_PROCESSED / 'featured_logs.parquet'
csv_path = P5_PROCESSED / 'featured_logs.csv'

if pq_path.exists():
    df = pd.read_parquet(pq_path)
else:
    df = pd.read_csv(csv_path)

print(f"Ingested Dataset: {len(df):,} records, {len(df.columns)} columns")
print("\nRaw Class Distribution:")
print(df['label'].value_counts())

# Stratified 80/20 Train/Test Split on indices
train_idx, test_idx = train_test_split(
    np.arange(len(df)),
    test_size=0.20,
    random_state=42,
    stratify=df['label'].values
)

df_train = df.iloc[train_idx].copy().reset_index(drop=True)
df_test = df.iloc[test_idx].copy().reset_index(drop=True)

print(f"\nTraining Partition (80%) : {len(df_train):,} records (Isolated for Resampling)")
print(f"Test Partition (20%)     : {len(df_test):,} records (LOCKED UNTOUCHED)")

# Export locked test partition
df_test.to_csv(PROCESSED_DIR / 'test_dataset.csv', index=False)
df_test.to_parquet(PROCESSED_DIR / 'test_dataset.parquet', index=False, compression='snappy')
print("Test partition successfully locked and exported.")

---
## 5. RESAMPLING BENCHMARK EVALUATION (RUS vs ROS vs SMOTE)
We evaluate 3 resampling techniques strictly on the training partition.

In [ ]:
X_train = np.ascontiguousarray(df_train[FEATURE_COLUMNS].fillna(0).to_numpy(dtype=np.float32))
y_train = np.ascontiguousarray(df_train['label'].to_numpy())
classes = np.unique(y_train)
orig_counts = pd.Series(y_train).value_counts().to_dict()
min_count = min(orig_counts.values())

# 1. Random Under-Sampling (RUS)
t0 = time.time()
rus = RandomUnderSampler(sampling_strategy={c: min_count for c in classes}, random_state=42)
X_rus, y_rus = rus.fit_resample(X_train, y_train)
t_rus = time.time() - t0
print(f"[RUS] Finished in {t_rus:.2f}s! Records: {len(X_rus):,}")

# 2. Random Over-Sampling (ROS)
TARGET_PER_CLASS = 15000
t0 = time.time()
ros_strategy = {c: TARGET_PER_CLASS for c in classes if orig_counts[c] < TARGET_PER_CLASS}
ros = RandomOverSampler(sampling_strategy=ros_strategy, random_state=42)
X_ros, y_ros = ros.fit_resample(X_train, y_train)
t_ros = time.time() - t0
print(f"[ROS] Finished in {t_ros:.2f}s! Records: {len(X_ros):,}")

# 3. SMOTE (k-NN Manifold Interpolation)
t0 = time.time()
smote_strategy = {c: TARGET_PER_CLASS for c in classes if orig_counts[c] < TARGET_PER_CLASS}
smote = SMOTE(sampling_strategy=smote_strategy, random_state=42, k_neighbors=5)
X_smote, y_smote = smote.fit_resample(X_train, y_train)
t_smote = time.time() - t0
print(f"[SMOTE] Finished in {t_smote:.2f}s! Records: {len(X_smote):,}")

---
## 6. FINAL BALANCED TRAINING DATASET SYNTHESIS
We assemble the primary balanced training matrix where all 11 classes contain exactly 15,000 records (165,000 total records).

In [ ]:
# Equalize benign to 15,000 + keep all 10 attack classes synthesized to 15,000
np.random.seed(42)
benign_idx = np.where(y_smote == 'benign')[0]
benign_sub = np.random.choice(benign_idx, size=TARGET_PER_CLASS, replace=False)

equal_indices = list(benign_sub)
for c in classes:
    if c != 'benign':
        equal_indices.extend(np.where(y_smote == c)[0])

equal_indices = np.array(equal_indices)
np.random.shuffle(equal_indices)

X_final = X_smote[equal_indices]
y_final = y_smote[equal_indices]

df_final_balanced = pd.DataFrame(X_final, columns=FEATURE_COLUMNS)
df_final_balanced['label'] = y_final

print(f"Final Equalized Training Dataset: {len(df_final_balanced):,} rows")
print(df_final_balanced['label'].value_counts())

# Export to CSV and Snappy-compressed Parquet
out_final_csv = PROCESSED_DIR / 'balanced_training_dataset.csv'
out_final_pq = PROCESSED_DIR / 'balanced_training_dataset.parquet'
df_final_balanced.to_csv(out_final_csv, index=False)
df_final_balanced.to_parquet(out_final_pq, index=False, compression='snappy')
print(f"Exported: {out_final_csv.name} ({out_final_csv.stat().st_size / (1024*1024):.2f} MB)")
print(f"Exported: {out_final_pq.name} ({out_final_pq.stat().st_size / (1024*1024):.2f} MB)")

---
## 7. PUBLICATION-QUALITY DIAGNOSTIC VISUALIZATIONS
We visualize the pre-balancing imbalance, train vs test alignment, balancing comparisons, and 2D PCA SMOTE manifolds.

In [ ]:
# 1. Original Imbalance Distribution
plt.figure(figsize=(10, 5), dpi=150)
counts = df_train['label'].value_counts()
colors = ['#10B981' if c == 'benign' else '#EF4444' for c in counts.index]
plt.barh(counts.index, counts.values, color=colors, edgecolor='#0F172A')
plt.title('Training Partition Class Distribution (80% Split)', fontsize=12, fontweight='bold')
plt.xlabel('Records')
plt.grid(True, axis='x', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

# 2. Balancing Techniques Comparison
plt.figure(figsize=(11, 5), dpi=150)
cls_list = sorted(classes)
x = np.arange(len(cls_list))
w = 0.2
plt.bar(x - 1.5*w, [orig_counts.get(c, 0) for c in cls_list], w, label='Original Train', color='#94A3B8')
plt.bar(x - 0.5*w, [pd.Series(y_rus).value_counts().get(c, 0) for c in cls_list], w, label='RUS', color='#EF4444')
plt.bar(x + 0.5*w, [pd.Series(y_ros).value_counts().get(c, 0) for c in cls_list], w, label='ROS', color='#F59E0B')
plt.bar(x + 1.5*w, [pd.Series(y_final).value_counts().get(c, 0) for c in cls_list], w, label='Final SMOTE', color='#10B981')
plt.xticks(x, cls_list, rotation=35, ha='right')
plt.yscale('log')
plt.title('Balancing Methods Comparison (Log Scale)', fontsize=12, fontweight='bold')
plt.ylabel('Records')
plt.legend()
plt.grid(True, axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()